In [39]:
import rectanglepy as rp
import pandas as pd
from anndata import AnnData, read_h5ad
import os
from typing import Dict, List, Tuple, Optional
import anndata as ad 

In [13]:
data_path='/mnt/cold1/snaketree/prj/scRNA/dataset/rCASC_Ire_cetuxi/CRC0322_cetux_1_dir/filtered_annotated_CRC0322_cetux_1.tsv'
data=pd.read_csv(data_path,header=0,index_col=0).T

In [9]:
def strip_prefix_from_genes(
    df: pd.DataFrame,
    meta_cols: Tuple[str, ...] = ("cell_id","sample","trattamento"),
    sep: str = ":",
    on_duplicate: str = "first",  # "error" | "first" | "mean" | "sum" | "suffix"
) -> Tuple[pd.DataFrame, Dict[str, List[str]]]:
    """
    Rename gene columns by removing the prefix before `sep`.
    Handle duplicates according to `on_duplicate` policy.
    Returns: (df_out, dup_report) where dup_report maps gene -> original columns
    """
    df = df.copy()
    meta_cols = list(meta_cols)
    gene_cols = [c for c in df.columns if c not in meta_cols]

    # map old gene col -> symbol after sep
    new_names = {c: c.split(sep)[-1] for c in gene_cols}
    df_ren = df.rename(columns=new_names)

    # build duplicate report
    dup_report: Dict[str, List[str]] = {}
    counts = pd.Series([new_names[c] for c in gene_cols]).value_counts()
    dups = counts[counts > 1].index.tolist()
    if dups:
        for g in dups:
            dup_report[g] = [c for c in gene_cols if new_names[c] == g]

        if on_duplicate == "error":
            raise ValueError(f"Duplicate genes after renaming: {dup_report}")

        elif on_duplicate in ("mean","sum"):
            agg_func = np.nanmean if on_duplicate == "mean" else np.nansum
            gdf = df_ren.drop(columns=meta_cols, errors="ignore")
            # collapse duplicates by aggregating columns with same name
            collapsed = {}
            for g, _cols in dup_report.items():
                mask = gdf.columns == g
                if mask.sum() > 1:
                    collapsed[g] = agg_func(gdf.loc[:, mask].to_numpy(), axis=1)
                else:
                    collapsed[g] = gdf.loc[:, g].to_numpy()
            keep_mask = ~gdf.columns.duplicated(keep=False)
            gdf_unique = gdf.loc[:, keep_mask].copy()
            for g, vec in collapsed.items():
                gdf_unique[g] = vec
            df_out = pd.concat([df_ren.loc[:, meta_cols], gdf_unique], axis=1)

        elif on_duplicate == "first":
            cols_meta = list(meta_cols)
            cols_gene_unique = ~df_ren.columns.duplicated(keep="first")
            df_out = pd.concat([df_ren.loc[:, cols_meta],
                                df_ren.loc[:, cols_gene_unique & ~df_ren.columns.isin(cols_meta)]],
                               axis=1)
        elif on_duplicate == "suffix":
            seen = {}
            newcols = []
            for c in df_ren.columns:
                if c in meta_cols:
                    newcols.append(c); continue
                name = c
                if name not in seen:
                    seen[name] = 1
                    newcols.append(name)
                else:
                    newcols.append(f"{name}.{seen[name]}")
                    seen[name] += 1
            df_out = df_ren.copy()
            df_out.columns = newcols
        else:
            raise ValueError(f"Unsupported on_duplicate={on_duplicate}")
    else:
        df_out = df_ren

    # reorder: meta first
    gene_cols_new = [c for c in df_out.columns if c not in meta_cols]
    df_out = pd.concat([df_out.loc[:, meta_cols], df_out.loc[:, gene_cols_new]], axis=1)

    return df_out, dup_report

In [14]:
trattamento='cetux'
data['sample']='CRC0322_cetux'
data['cell_id']=data.index
data['trattamento']=trattamento
data.reset_index(drop=True,inplace=True)
df_clean, dup = strip_prefix_from_genes(data, meta_cols=("cell_id","sample","trattamento"), sep=":", on_duplicate="first")

In [21]:
clustering_path='/mnt/cold1/snaketree/prj/scRNA/dataset/GMM/posterior/CRC0322_cetux_1/CRC0322_cetux_1_posterior_full_3comp.csv'
clust=pd.read_csv(clustering_path,index_col=0,header=0)
clust=clust[clust['isPaneth']!='Medium']

In [27]:
cell_type=clust[['isPaneth']]
cell_type.rename(columns={'isPaneth':'cell_type'},inplace=True)

/tmp/ipykernel_396746/2421819853.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cell_type.rename(columns={'isPaneth':'cell_type'},inplace=True)


In [31]:
cell_type_series = cell_type["cell_type"].copy()
df = df_clean.copy()
df = df.set_index("cell_id")
meta_cols = ["sample", "trattamento"]
gene_df = df.drop(columns=meta_cols, errors="ignore")
common_cells = gene_df.index.intersection(cell_type_series.index)

print("Cellule in gene_df:", gene_df.shape[0])
print("Cellule in cell_type:", cell_type_series.shape[0])
print("Cellule in comune:", len(common_cells))

Cellule in gene_df: 2466
Cellule in cell_type: 2065
Cellule in comune: 2065


In [36]:
gene_df = gene_df.loc[common_cells].copy()
cell_type_series = cell_type_series.loc[common_cells].copy()
adata = ad.AnnData(X=gene_df)
adata.obs["cell_type"] = cell_type_series

In [44]:
sig = rp.pp.build_rectangle_signatures(
    adata,
    cell_type_col="cell_type",
    gene_expression_threshold=0.3
)

2026-04-21 14:44:22.915 | INFO     | rectanglepy.pp.create_signature:_de_analysis:200 - Starting DE analysis
2026-04-21 14:44:24.369 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:148 - Running DE analysis for High
2026-04-21 14:44:28.909 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:148 - Running DE analysis for Low
2026-04-21 14:44:34.769 | INFO     | rectanglepy.pp.create_signature:_de_analysis:205 - Optimizing cutoff parameters p and lfc
2026-04-21 14:44:34.771 | INFO     | rectanglepy.pp.create_signature:_optimize_parameters:408 - generating pseudo bulks
2026-04-21 14:44:36.475 | INFO     | rectanglepy.pp.create_signature:_get_marker_genes:225 - Optimal condition number: 25


Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optima

2026-04-21 14:44:36.753 | INFO     | rectanglepy.pp.create_signature:_optimize_parameters:416 - RMSE:0.15462803102426426, Pearson R:0.835366690638202 for p=0.05, lfc=1.6
2026-04-21 14:44:36.766 | INFO     | rectanglepy.pp.create_signature:_get_marker_genes:225 - Optimal condition number: 25


Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point


2026-04-21 14:44:37.040 | INFO     | rectanglepy.pp.create_signature:_optimize_parameters:416 - RMSE:0.15462803102426426, Pearson R:0.835366690638202 for p=0.05, lfc=1.7
2026-04-21 14:44:37.050 | INFO     | rectanglepy.pp.create_signature:_get_marker_genes:225 - Optimal condition number: 23


...

Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point


2026-04-21 14:44:38.174 | INFO     | rectanglepy.pp.create_signature:_optimize_parameters:416 - RMSE:0.2156749930887894, Pearson R:0.7660602064356609 for p=0.05, lfc=2.1
2026-04-21 14:44:38.183 | INFO     | rectanglepy.pp.create_signature:_get_marker_genes:225 - Optimal condition number: 21


Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point


2026-04-21 14:44:38.453 | INFO     | rectanglepy.pp.create_signature:_optimize_parameters:416 - RMSE:0.2156749930887894, Pearson R:0.7660602064356609 for p=0.05, lfc=2.2
2026-04-21 14:44:38.458 | INFO     | rectanglepy.pp.create_signature:_de_analysis:208 - Optimization done
 Best cutoffs  p: 0.05 and lfc: 1.6
2026-04-21 14:44:38.471 | INFO     | rectanglepy.pp.create_signature:_get_marker_genes:225 - Optimal condition number: 25
2026-04-21 14:44:38.475 | INFO     | rectanglepy.pp.create_signature:build_rectangle_signatures:348 - Starting rectangle cluster analysis
2026-04-21 14:44:38.477 | INFO     | rectanglepy.pp.create_signature:_create_clustered_data:255 - Not enough cell types to perform clustering, returning direct rectangle signature


Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point


In [45]:
sig

RectangleSignatureResult
──────────────────────────────────────────────────
  Total Signature genes: 25
  Cell types: 2
    High marker genes: ['ANXA13', 'ARFGEF3', 'ID2', 'ELAPOR1', 'SMAD9', 'SOX4', 'FOXA2', 'RNASE1', 'IL13RA1', 'MICAL1', 'KLF4', 'AMIGO2', 'GAS2L3', 'HES6', 'TFF3', 'RETNLB', 'DEFA6', 'TUBA1A', 'KLK11', 'RAB26', 'ATOH1', 'RGMB', 'PTCH1', 'HEPACAM2', 'ENTPD8']
    Low marker genes: []
  DGE Optimization result available:
    Best cutoffs: p: 0.05, lfc: 1.6

In [46]:
tests = [
    {"p": 0.05, "lfc": 1.0},
    {"p": 0.1,  "lfc": 1},
]

results = []

for pars in tests:
    sig_tmp = rp.pp.build_rectangle_signatures(
        adata,
        cell_type_col="cell_type",
        optimize_cutoffs=False,
        p=pars["p"],
        lfc=pars["lfc"],
        gene_expression_threshold=0.3
    )
    
    results.append({
        "p": pars["p"],
        "lfc": pars["lfc"],
        "n_total": len(sig_tmp.signature_genes),
        "markers_per_type": sig_tmp.marker_genes_per_cell_type
    })

results

2026-04-21 15:00:18.532 | INFO     | rectanglepy.pp.create_signature:_de_analysis:200 - Starting DE analysis
2026-04-21 15:00:32.209 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:148 - Running DE analysis for High
2026-04-21 15:00:41.318 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:148 - Running DE analysis for Low
2026-04-21 15:00:47.314 | INFO     | rectanglepy.pp.create_signature:_get_marker_genes:225 - Optimal condition number: 76
2026-04-21 15:00:47.319 | INFO     | rectanglepy.pp.create_signature:build_rectangle_signatures:348 - Starting rectangle cluster analysis
2026-04-21 15:00:47.320 | INFO     | rectanglepy.pp.create_signature:_create_clustered_data:255 - Not enough cell types to perform clustering, returning direct rectangle signature
2026-04-21 15:00:54.529 | INFO     | rectanglepy.pp.create_signature:_de_analysis:200 - Starting DE analysis
2026-04-21 15:01:08.787 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:148 - Running DE analysis for 

[{'p': 0.05,
  'lfc': 1.0,
  'n_total': 78,
  'markers_per_type': {'High': ['GOPC',
    'ST6GALNAC1',
    'MAP4K4',
    'NTN4',
    'CBFA2T2',
    'CDHR5',
    'XBP1',
    'ANXA13',
    'OLFM2',
    'LFNG',
    'CHN2',
    'ZKSCAN1',
    'GLCCI1',
    'TSPAN14',
    'ABCC3',
    'PRKAR1A',
    'MDK',
    'MAN1A1',
    'CRYBG1',
    'ARFGEF3',
    'TENT5A',
    'ID2',
    'HPCAL1',
    'ELAPOR1',
    'PROX1',
    'HOXB3',
    'SMAD9',
    'PRXL2A',
    'ODF2L',
    'SOX4',
    'FOXA2',
    'ID1',
    'RNASE1',
    'GSE1',
    'IL13RA1',
    'SLC38A2',
    'ARHGAP32',
    'MICAL1',
    'GCC2',
    'CKAP4',
    'KLF4',
    'SLC22A23',
    'AMIGO2',
    'GAS2L3',
    'TPM1',
    'ARRDC4',
    'HES6',
    'SESN3',
    'BTG2',
    'TFF3',
    'IHH',
    'RETNLB',
    'SPINK1',
    'DEFA6',
    'TP53INP1',
    'BLCAP',
    'TUBA1A',
    'KLK11',
    'HID1',
    'RAB26',
    'FOXA3',
    'GSTA4',
    'INSR',
    'ATOH1',
    'MOB1B',
    'RGMB',
    'B3GNT5',
    'PTCH1',
    'HEPACAM2',
    '

In [48]:
umap_path='/mnt/cold2/snaketree/prj/PPH/local/share/data/integrated_umap/CRC0322_cetux_1_umap_integrated.csv'
umap=pd.read_csv(umap_path)